In [1]:
# ── Imports ───────────────────────────────────────────────────────────────────
import requests
import pandas as pd
import os
import json
from datetime import datetime

# Suppress the urllib3 SSL warning we saw earlier - it's harmless
import warnings
warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
# Build paths relative to this notebook's location so they work on any machine
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
RAW_DIR      = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# Create directories if they don't exist
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Raw data folder:       {RAW_DIR}")
print(f"Processed data folder: {PROCESSED_DIR}")
print(f"Setup complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

/Users/paridhisingh/Desktop/growing-up-in-sa/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Raw data folder:       /Users/paridhisingh/Desktop/growing-up-in-sa/data/raw
Processed data folder: /Users/paridhisingh/Desktop/growing-up-in-sa/data/processed
Setup complete: 2026-04-09 12:31:17


/var/folders/bx/6qc4yg5x1rdf3qbnj7y1dnb80000gn/T/ipykernel_49169/3508681408.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# ── ABS API - Test Connection ─────────────────────────────────────────────────
BASE_URL = "https://api.data.abs.gov.au"

response = requests.get(
    f"{BASE_URL}/data/ABS_CENSUS2021_B04/all/ABS"
    "?startPeriod=2021&endPeriod=2021"
    "&dimensionAtObservation=AllDimensions"
    "&format=jsondata"
)

print(f"Status code: {response.status_code}")
if response.status_code == 200:
    print("✅ ABS API is reachable")
else:
    print(f"❌ Something went wrong - response: {response.text[:200]}")

Status code: 404
❌ Something went wrong - response: Could not find Dataflow and/or DSD related with this data request


In [3]:
# ── Load ABS Census G04 - Age by Sex by LGA ───────────────────────────────────
import glob

# Path to the Census CSV folder
CENSUS_DIR = os.path.join(
    RAW_DIR,
    "2021_GCP_LGA_for_SA_short-header",
    "2021 Census GCP Local Government Areas for SA"
)

# ── Load G04A and G04B (age data is split across two files) ───────────────────
g04a_path = os.path.join(CENSUS_DIR, "2021Census_G04A_SA_LGA.csv")
g04b_path = os.path.join(CENSUS_DIR, "2021Census_G04B_SA_LGA.csv")

g04a = pd.read_csv(g04a_path)
g04b = pd.read_csv(g04b_path)

# Merge the two files on LGA code
g04 = pd.merge(g04a, g04b, on="LGA_CODE_2021")

print(f"G04A shape: {g04a.shape}")
print(f"G04B shape: {g04b.shape}")
print(f"Combined shape: {g04.shape}")
print(f"\nNumber of LGAs: {len(g04)}")
print(f"\nFirst few LGA codes:")
print(g04["LGA_CODE_2021"].head())


G04A shape: (73, 201)
G04B shape: (73, 107)
Combined shape: (73, 307)

Number of LGAs: 73

First few LGA codes:
0    LGA40070
1    LGA40120
2    LGA40150
3    LGA40220
4    LGA40250
Name: LGA_CODE_2021, dtype: object


In [4]:
# ── Extract only the age groups relevant to our project ───────────────────────
# We care about 0-24 year olds (children and young people)
# We'll use the pre-aggregated 5-year bands ending in _P (persons = male + female)

age_band_cols = [
    "LGA_CODE_2021",
    "Age_yr_0_4_P",
    "Age_yr_5_9_P",
    "Age_yr_10_14_P",
    "Age_yr_15_19_P",
    "Age_yr_20_24_P",
]

# Subset to just these columns
df_age = g04[age_band_cols].copy()

# Rename for readability
df_age.columns = [
    "lga_code",
    "age_0_4",
    "age_5_9",
    "age_10_14",
    "age_15_19",
    "age_20_24",
]

# ── Reshape from wide to long format ──────────────────────────────────────────
df_long = df_age.melt(
    id_vars="lga_code",
    var_name="age_group",
    value_name="population"
)

# Add a year column for future time-series joins
df_long["census_year"] = 2021

# Clean up age_group labels
df_long["age_group"] = df_long["age_group"].str.replace("age_", "").str.replace("_", "-")

print(df_long.head(15))
print(f"\nShape: {df_long.shape}")
print(f"\nAge groups: {df_long['age_group'].unique()}")
print(f"LGAs: {df_long['lga_code'].nunique()}")

    lga_code age_group  population  census_year
0   LGA40070       0-4         503         2021
1   LGA40120       0-4        1978         2021
2   LGA40150       0-4         564         2021
3   LGA40220       0-4        1120         2021
4   LGA40250       0-4         197         2021
5   LGA40310       0-4        1182         2021
6   LGA40430       0-4         115         2021
7   LGA40520       0-4         444         2021
8   LGA40700       0-4        1830         2021
9   LGA40910       0-4        2736         2021
10  LGA41010       0-4         234         2021
11  LGA41060       0-4        6264         2021
12  LGA41140       0-4         444         2021
13  LGA41190       0-4         106         2021
14  LGA41330       0-4          67         2021

Shape: (365, 4)

Age groups: ['0-4' '5-9' '10-14' '15-19' '20-24']
LGAs: 73


In [6]:
# ── Load LGA reference file from Metadata folder ──────────────────────────────
metadata_path = os.path.join(
    RAW_DIR,
    "2021_GCP_LGA_for_SA_short-header",
    "Metadata",
    "2021Census_geog_desc_1st_2nd_3rd_release.xlsx"
)

# Load all sheets first so we can see what's available
lga_ref = pd.read_excel(metadata_path, sheet_name=None)

print("Sheets in metadata file:")
for sheet in lga_ref.keys():
    print(f"  - {sheet}")

Sheets in metadata file:
  - 2021_ASGS_MAIN_Structures
  - 2021_ASGS_GCCSA_Structure
  - 2021_ASGS_Indigenous_Structures
  - 2021_ASGS_SOS_UCL_Structures
  - 2021_ASGS_SUA_Structures
  - 2021_ASGS_Non_ABS_Structures
  - 2021_ASGS_RA_Structures


In [7]:
# ── Peek inside the Non-ABS structures sheet ──────────────────────────────────
non_abs = lga_ref["2021_ASGS_Non_ABS_Structures"]

print(f"Shape: {non_abs.shape}")
print(f"\nColumns: {non_abs.columns.tolist()}")
print(f"\nFirst 10 rows:")
print(non_abs.head(10))

Shape: (19181, 5)

Columns: ['ASGS_Structure', 'Census_Code_2021', 'AGSS_Code_2021', 'Census_Name_2021', 'Area sqkm']

First 10 rows:
  ASGS_Structure Census_Code_2021 AGSS_Code_2021 Census_Name_2021  \
0            AUS              AUS            AUS        AUSTRALIA   
1            CED           CED101            101            Banks   
2            CED           CED102            102           Barton   
3            CED           CED103            103        Bennelong   
4            CED           CED104            104          Berowra   
5            CED           CED105            105         Blaxland   
6            CED           CED106            106        Bradfield   
7            CED           CED107            107           Calare   
8            CED           CED108            108          Chifley   
9            CED           CED109            109             Cook   

      Area sqkm  
0  7.688095e+06  
1  4.943710e+01  
2  3.954700e+01  
3  5.869100e+01  
4  7.439867e+02 

In [8]:
# ── Filter to SA LGAs only ────────────────────────────────────────────────────
lga_lookup = non_abs[non_abs["ASGS_Structure"] == "LGA"].copy()

print(f"Total LGAs nationally: {len(lga_lookup)}")
print(f"\nFirst 10 rows:")
print(lga_lookup.head(10))

# Check what SA LGA codes look like - we know ours start with LGA4
sa_lgas = lga_lookup[lga_lookup["Census_Code_2021"].str.startswith("LGA4")]
print(f"\nSA LGAs found: {len(sa_lgas)}")
print(sa_lgas.head(10))

Total LGAs nationally: 565

First 10 rows:
    ASGS_Structure Census_Code_2021 AGSS_Code_2021   Census_Name_2021  \
621            LGA         LGA10050          10050             Albury   
622            LGA         LGA10180          10180  Armidale Regional   
623            LGA         LGA10250          10250            Ballina   
624            LGA         LGA10300          10300          Balranald   
625            LGA         LGA10470          10470  Bathurst Regional   
626            LGA         LGA10500          10500      Bayside (NSW)   
627            LGA         LGA10550          10550        Bega Valley   
628            LGA         LGA10600          10600          Bellingen   
629            LGA         LGA10650          10650           Berrigan   
630            LGA         LGA10750          10750          Blacktown   

      Area sqkm  
621    305.6386  
622   7809.4406  
623    484.9692  
624  21690.7493  
625   3817.8645  
626     50.6204  
627   6278.5013  
628   160

In [9]:
# ── Join LGA names to our population data ─────────────────────────────────────
# Rename columns for the join
lga_lookup_sa = sa_lgas[["Census_Code_2021", "Census_Name_2021", "Area sqkm"]].copy()
lga_lookup_sa.columns = ["lga_code", "lga_name", "area_sqkm"]

# Join to our long format dataframe
df_final = pd.merge(df_long, lga_lookup_sa, on="lga_code", how="left")

# Reorder columns neatly
df_final = df_final[["lga_code", "lga_name", "area_sqkm", "census_year", "age_group", "population"]]

# Quick check
print(df_final.head(10))
print(f"\nShape: {df_final.shape}")
print(f"\nAny missing LGA names? {df_final['lga_name'].isna().sum()}")
print(f"\nSample LGAs:")
print(df_final["lga_name"].unique()[:10])

   lga_code                               lga_name    area_sqkm  census_year  \
0  LGA40070                               Adelaide      15.5733         2021   
1  LGA40120                         Adelaide Hills     794.4957         2021   
2  LGA40150                        Adelaide Plains     932.4913         2021   
3  LGA40220                            Alexandrina    1826.8067         2021   
4  LGA40250  Anangu Pitjantjatjara Yunkunytjatjara  102348.1170         2021   
5  LGA40310                                Barossa     893.5424         2021   
6  LGA40430                           Barunga West    1590.3887         2021   
7  LGA40520                          Berri Barmera     476.1962         2021   
8  LGA40700                               Burnside      27.5183         2021   
9  LGA40910                      Campbelltown (SA)      24.3485         2021   

  age_group  population  
0       0-4         503  
1       0-4        1978  
2       0-4         564  
3       0-4    

In [10]:
# ── Save to processed folder ───────────────────────────────────────────────────
output_path = os.path.join(PROCESSED_DIR, "abs_demographics_sa_2021.csv")
df_final.to_csv(output_path, index=False)

print(f"✅ File saved to: {output_path}")
print(f"\nFinal dataset summary:")
print(f"  Rows:        {len(df_final)}")
print(f"  LGAs:        {df_final['lga_code'].nunique()}")
print(f"  Age groups:  {sorted(df_final['age_group'].unique())}")
print(f"  Year:        {df_final['census_year'].unique()}")
print(f"  Total 0-24 population in SA: {df_final['population'].sum():,}")
print(f"\nTop 10 LGAs by 0-24 population:")
top10 = df_final.groupby("lga_name")["population"].sum().sort_values(ascending=False).head(10)
print(top10.to_string())

✅ File saved to: /Users/paridhisingh/Desktop/growing-up-in-sa/data/processed/abs_demographics_sa_2021.csv

Final dataset summary:
  Rows:        365
  LGAs:        73
  Age groups:  ['0-4', '10-14', '15-19', '20-24', '5-9']
  Year:        [2021]
  Total 0-24 population in SA: 511,540

Top 10 LGAs by 0-24 population:
lga_name
Onkaparinga              51149
Salisbury                47292
Port Adelaide Enfield    37771
Playford                 35806
Charles Sturt            32337
Tea Tree Gully           29212
Marion                   26455
Mitcham                  20036
West Torrens             16683
Campbelltown (SA)        15367


In [11]:
# ── Load G07 - Indigenous Status by Age by LGA ────────────────────────────────
g07_path = os.path.join(CENSUS_DIR, "2021Census_G07_SA_LGA.csv")
g07 = pd.read_csv(g07_path)

print(f"Shape: {g07.shape}")
print(f"LGAs: {len(g07)}")

# ── Extract Indigenous population for 0-24 age groups only ───────────────────
# We want the _P (persons) columns for Indigenous only, for each age band
indigenous_cols = [
    "LGA_CODE_2021",
    "A_0_4_y_Indigenous_P",
    "A_5_9_y_Indigenous_P",
    "A_10_14_y_Indigenous_P",
    "A_15_19_y_Indigenous_P",
    "A_20_24_y_Indigenous_P",
]

df_indig = g07[indigenous_cols].copy()

# Rename columns
df_indig.columns = [
    "lga_code",
    "age_0_4",
    "age_5_9",
    "age_10_14",
    "age_15_19",
    "age_20_24",
]

# ── Reshape to long format ────────────────────────────────────────────────────
df_indig_long = df_indig.melt(
    id_vars="lga_code",
    var_name="age_group",
    value_name="indigenous_population"
)

# Clean up age group labels
df_indig_long["age_group"] = df_indig_long["age_group"].str.replace("age_", "").str.replace("_", "-")
df_indig_long["census_year"] = 2021

print(df_indig_long.head(10))
print(f"\nShape: {df_indig_long.shape}")

Shape: (73, 181)
LGAs: 73
   lga_code age_group  indigenous_population  census_year
0  LGA40070       0-4                      8         2021
1  LGA40120       0-4                     17         2021
2  LGA40150       0-4                     39         2021
3  LGA40220       0-4                     40         2021
4  LGA40250       0-4                    189         2021
5  LGA40310       0-4                     30         2021
6  LGA40430       0-4                     10         2021
7  LGA40520       0-4                     39         2021
8  LGA40700       0-4                      8         2021
9  LGA40910       0-4                     39         2021

Shape: (365, 4)


In [12]:
# ── Merge Indigenous data with main demographics dataframe ────────────────────
df_combined = pd.merge(
    df_final,
    df_indig_long[["lga_code", "age_group", "indigenous_population"]],
    on=["lga_code", "age_group"],
    how="left"
)

# ── Calculate Indigenous percentage ───────────────────────────────────────────
df_combined["indigenous_pct"] = (
    df_combined["indigenous_population"] / df_combined["population"] * 100
).round(2)

# Reorder columns
df_combined = df_combined[[
    "lga_code",
    "lga_name", 
    "area_sqkm",
    "census_year",
    "age_group",
    "population",
    "indigenous_population",
    "indigenous_pct"
]]

# ── Quick sense check ─────────────────────────────────────────────────────────
print(df_combined.head(10))
print(f"\nShape: {df_combined.shape}")
print(f"\nAny nulls? {df_combined.isnull().sum().sum()}")
print(f"\nTop 10 LGAs by Indigenous % (0-24 population):")
top_indig = (
    df_combined.groupby("lga_name")[["population", "indigenous_population"]]
    .sum()
    .assign(indigenous_pct=lambda x: (x["indigenous_population"] / x["population"] * 100).round(2))
    .sort_values("indigenous_pct", ascending=False)
    .head(10)
)
print(top_indig.to_string())

   lga_code                               lga_name    area_sqkm  census_year  \
0  LGA40070                               Adelaide      15.5733         2021   
1  LGA40120                         Adelaide Hills     794.4957         2021   
2  LGA40150                        Adelaide Plains     932.4913         2021   
3  LGA40220                            Alexandrina    1826.8067         2021   
4  LGA40250  Anangu Pitjantjatjara Yunkunytjatjara  102348.1170         2021   
5  LGA40310                                Barossa     893.5424         2021   
6  LGA40430                           Barunga West    1590.3887         2021   
7  LGA40520                          Berri Barmera     476.1962         2021   
8  LGA40700                               Burnside      27.5183         2021   
9  LGA40910                      Campbelltown (SA)      24.3485         2021   

  age_group  population  indigenous_population  indigenous_pct  
0       0-4         503                      8        

In [13]:
# ── Investigate the 5 nulls ───────────────────────────────────────────────────
print("Rows with null values:")
print(df_combined[df_combined.isnull().any(axis=1)])

Rows with null values:
     lga_code                              lga_name  area_sqkm  census_year  \
72   LGA49799  Migratory - Offshore - Shipping (SA)        0.0         2021   
145  LGA49799  Migratory - Offshore - Shipping (SA)        0.0         2021   
218  LGA49799  Migratory - Offshore - Shipping (SA)        0.0         2021   
291  LGA49799  Migratory - Offshore - Shipping (SA)        0.0         2021   
364  LGA49799  Migratory - Offshore - Shipping (SA)        0.0         2021   

    age_group  population  indigenous_population  indigenous_pct  
72        0-4           0                      0             NaN  
145       5-9           0                      0             NaN  
218     10-14           0                      0             NaN  
291     15-19           0                      0             NaN  
364     20-24           0                      0             NaN  
